# Kernel Approximation pro klasifikaci

## Co je Kernel Approximation?

Kernel approximation (aproximace jádra) je technika, která umožňuje použít výhody jádrových metod bez jejich výpočetní náročnosti pro velké datové sady. Místo výpočtu plné jádrové matice, která může být výpočetně náročná pro velké datasety (složitost $O(n^2)$), kernel approximation transformuje původní příznaky do nového prostoru, kde lineární modely mohou dosáhnout podobných výsledků jako jádrové metody.

### Kdy použít Kernel Approximation:
- Pro velké datové sady, kde tradiční jádrové metody (jako SVM s RBF jádrem) jsou výpočetně neúnosné
- Když data nejsou lineárně oddělitelná, ale potřebujete efektivní škálovatelné řešení
- Pro online učení, kde data přicházejí postupně
- Jako součást pipeline s rychlými lineárními klasifikátory (např. SGDClassifier)

### Výhody:
- Lineární škálovatelnost s velikostí dat (namísto kvadratické u plných jádrových metod)
- Kompatibilita s lineárními modely, které jsou rychlé a efektivní
- Schopnost pracovat s velkými datovými sadami a online učením
- Zachovává mnoho výhod jádrových metod pro nelineární problémy

### Nevýhody:
- Aproximace může být méně přesná než plná jádrová metoda
- Vyžaduje pečlivé nastavení parametrů (např. počet komponent, parametry jádra)
- Přidává další hyperparametry, které je třeba optimalizovat

V tomto notebooku se zaměříme hlavně na RBFSampler, který aproximuje Gaussovské RBF jádro. Existují i další techniky jako Nystroem nebo SkewedChi2Sampler pro jiné typy jader.

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import RBFSampler, Nystroem
from sklearn.svm import SVC
from time import time

# Nastavení pro reprodukovatelnost výsledků
np.random.seed(42)

# Nastavení pro lepší vizualizaci
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Příprava dat pro demonstraci

Pro demonstraci kernel approximation použijeme několik syntetických datasetů, které jsou nelineárně oddělitelné, a jeden reálný dataset.

In [ ]:
# Generování syntetických datasetů
# Dataset ve tvaru dvou půlměsíců
X_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=42)

# Dataset ve tvaru soustředných kružnic
X_circles, y_circles = make_circles(n_samples=1000, noise=0.1, factor=0.5, random_state=42)

# Načtení reálného datasetu - Breast Cancer Wisconsin dataset
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

# Rozdělení dat na trénovací a testovací množiny
X_moons_train, X_moons_test, y_moons_train, y_moons_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=42)

X_circles_train, X_circles_test, y_circles_train, y_circles_test = train_test_split(
    X_circles, y_circles, test_size=0.2, random_state=42)

X_cancer_train, X_cancer_test, y_cancer_train, y_cancer_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42)

print("Tvar trénovacích dat - moons:", X_moons_train.shape)
print("Tvar trénovacích dat - circles:", X_circles_train.shape)
print("Tvar trénovacích dat - cancer:", X_cancer_train.shape)

### Vizualizace syntetických datasetů

Podívejme se na syntetické datasety, abychom lépe pochopili, proč by lineární klasifikátory měly problémy s těmito daty.

In [ ]:
# Funkce pro vizualizaci datových sad
def plot_dataset(X, y, title):
    plt.figure(figsize=(10, 6))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='k', alpha=0.7)
    plt.title(title, fontsize=15)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.colorbar(label='Třída')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Vizualizace dvou půlměsíců
plot_dataset(X_moons, y_moons, "Dataset: Dva půlměsíce")

# Vizualizace soustředných kružnic
plot_dataset(X_circles, y_circles, "Dataset: Soustředné kružnice")

## 2. Implementace kernel approximation

Nyní implementujeme klasifikaci pomocí kernel approximation a porovnáme ji s lineárním klasifikátorem a plnou jádrovou SVM.

In [ ]:
# Funkce pro vizualizaci rozhodovací hranice
def plot_decision_boundary(clf, X, y, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Vytvoření mřížky pro predikci
    h = 0.05  # krok mřížky
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predikce pro každý bod mřížky
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Vykreslení hranice rozhodování
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='k', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    return ax

### 2.1 Porovnání na datasetu půlměsíců

Nyní porovnáme tři přístupy na datasetu půlměsíců:
1. Lineární klasifikátor (SGDClassifier)
2. Kernel approximation s lineárním klasifikátorem
3. SVM s RBF jádrem

In [ ]:
# 1. Lineární klasifikátor
linear_clf = make_pipeline(StandardScaler(), SGDClassifier(max_iter=1000, tol=1e-3, random_state=42))

# 2. Kernel approximation s lineárním klasifikátorem
rbf_feature = RBFSampler(gamma=2, n_components=100, random_state=42)
rbf_pipeline = make_pipeline(
    StandardScaler(),
    rbf_feature,
    SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
)

# 3. SVM s RBF jádrem
svm_rbf = make_pipeline(StandardScaler(), SVC(kernel='rbf', gamma=2, random_state=42))

# Trénování modelů a měření času
models = [
    ("Lineární klasifikátor", linear_clf),
    ("Kernel approximation", rbf_pipeline),
    ("SVM s RBF jádrem", svm_rbf)
]

results = []

for name, model in models:
    start_time = time()
    model.fit(X_moons_train, y_moons_train)
    train_time = time() - start_time
    
    start_time = time()
    y_pred = model.predict(X_moons_test)
    predict_time = time() - start_time
    
    accuracy = accuracy_score(y_moons_test, y_pred)
    results.append((name, accuracy, train_time, predict_time))
    print(f"{name}:\n  Přesnost: {accuracy:.4f}\n  Čas trénování: {train_time:.4f}s\n  Čas predikce: {predict_time:.4f}s\n")

In [ ]:
# Vizualizace rozhodovacích hranic všech tří modelů
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (name, model) in enumerate(models):
    plot_decision_boundary(model, X_moons, y_moons, name, axes[i])

plt.tight_layout()
plt.show()

### 2.2 Porovnání na datasetu soustředných kružnic

Tento dataset představuje ještě větší výzvu pro lineární klasifikátor.

In [ ]:
# Znovu trénujeme modely na datasetu kružnic
circle_results = []

for name, model in models:
    start_time = time()
    model.fit(X_circles_train, y_circles_train)
    train_time = time() - start_time
    
    start_time = time()
    y_pred = model.predict(X_circles_test)
    predict_time = time() - start_time
    
    accuracy = accuracy_score(y_circles_test, y_pred)
    circle_results.append((name, accuracy, train_time, predict_time))
    print(f"{name}:\n  Přesnost: {accuracy:.4f}\n  Čas trénování: {train_time:.4f}s\n  Čas predikce: {predict_time:.4f}s\n")

In [ ]:
# Vizualizace rozhodovacích hranic všech tří modelů na datasetu kružnic
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (name, model) in enumerate(models):
    plot_decision_boundary(model, X_circles, y_circles, name, axes[i])

plt.tight_layout()
plt.show()

### 2.3 Výkon na reálném datasetu (Wisconsin Breast Cancer)

Nyní vyzkoušíme stejné modely na reálném datasetu s více dimenzemi.

In [ ]:
# Trénování a vyhodnocení na datasetu rakoviny prsu
cancer_results = []

for name, model in models:
    start_time = time()
    model.fit(X_cancer_train, y_cancer_train)
    train_time = time() - start_time
    
    start_time = time()
    y_pred = model.predict(X_cancer_test)
    predict_time = time() - start_time
    
    accuracy = accuracy_score(y_cancer_test, y_pred)
    cancer_results.append((name, accuracy, train_time, predict_time))
    print(f"{name}:\n  Přesnost: {accuracy:.4f}\n  Čas trénování: {train_time:.4f}s\n  Čas predikce: {predict_time:.4f}s")
    print(f"  Klasifikační report:\n{classification_report(y_cancer_test, y_pred)}\n")

In [ ]:
# Vizualizace matice záměn pro kernel approximation na datasetu rakoviny prsu
y_pred_cancer = rbf_pipeline.predict(X_cancer_test)
cm = confusion_matrix(y_cancer_test, y_pred_cancer)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predikovaná třída')
plt.ylabel('Skutečná třída')
plt.title('Matice záměn - Kernel Approximation (RBFSampler)')
plt.show()

## 3. Analýza vlivu počtu komponent

Jedním z klíčových parametrů RBFSampler je `n_components`, který určuje počet Monte Carlo vzorků (dimenzí) použitých pro aproximaci. Prozkoumejme, jak tento parametr ovlivňuje přesnost a efektivitu.

In [ ]:
# Testování různých počtů komponent
n_components_list = [10, 50, 100, 200, 500]
component_results = []

for n_comp in n_components_list:
    rbf_pipeline = make_pipeline(
        StandardScaler(),
        RBFSampler(gamma=2, n_components=n_comp, random_state=42),
        SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
    )
    
    start_time = time()
    rbf_pipeline.fit(X_moons_train, y_moons_train)
    train_time = time() - start_time
    
    start_time = time()
    y_pred = rbf_pipeline.predict(X_moons_test)
    predict_time = time() - start_time
    
    accuracy = accuracy_score(y_moons_test, y_pred)
    component_results.append((n_comp, accuracy, train_time, predict_time))
    
    print(f"n_components = {n_comp}:")
    print(f"  Přesnost: {accuracy:.4f}")
    print(f"  Čas trénování: {train_time:.4f}s")
    print(f"  Čas predikce: {predict_time:.4f}s\n")

In [ ]:
# Vizualizace výsledků
n_components = [res[0] for res in component_results]
accuracies = [res[1] for res in component_results]
train_times = [res[2] for res in component_results]
predict_times = [res[3] for res in component_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Graf přesnosti v závislosti na počtu komponent
ax1.plot(n_components, accuracies, 'o-', color='#2A9D8F')
ax1.set_xlabel('Počet komponent')
ax1.set_ylabel('Přesnost')
ax1.set_title('Vliv počtu komponent na přesnost')
ax1.grid(True)

# Graf času trénování v závislosti na počtu komponent
ax2.plot(n_components, train_times, 'o-', label='Čas trénování', color='#E9C46A')
ax2.plot(n_components, predict_times, 'o-', label='Čas predikce', color='#E76F51')
ax2.set_xlabel('Počet komponent')
ax2.set_ylabel('Čas [s]')
ax2.set_title('Vliv počtu komponent na výpočetní čas')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 4. Porovnání různých metod kernel approximation

Scikit-learn nabízí několik metod kernel approximation. Nyní porovnáme RBFSampler s metodou Nystroem.

In [ ]:
# Porovnání RBFSampler a Nystroem
rbf_sampler_pipeline = make_pipeline(
    StandardScaler(),
    RBFSampler(gamma=2, n_components=100, random_state=42),
    SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
)

nystroem_pipeline = make_pipeline(
    StandardScaler(),
    Nystroem(kernel='rbf', gamma=2, n_components=100, random_state=42),
    SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
)

# Trénování a vyhodnocení na datasetu půlměsíců
approximation_models = [
    ("RBFSampler", rbf_sampler_pipeline),
    ("Nystroem", nystroem_pipeline)
]

for name, model in approximation_models:
    start_time = time()
    model.fit(X_moons_train, y_moons_train)
    train_time = time() - start_time
    
    start_time = time()
    y_pred = model.predict(X_moons_test)
    predict_time = time() - start_time
    
    accuracy = accuracy_score(y_moons_test, y_pred)
    print(f"{name}:\n  Přesnost: {accuracy:.4f}\n  Čas trénování: {train_time:.4f}s\n  Čas predikce: {predict_time:.4f}s\n")

In [ ]:
# Vizualizace rozhodovacích hranic obou metod
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for i, (name, model) in enumerate(approximation_models):
    plot_decision_boundary(model, X_moons, y_moons, name, axes[i])

plt.tight_layout()
plt.show()

## 5. Optimalizace hyperparametrů pomocí GridSearchCV

Pro nalezení optimálních parametrů pro Kernel approximation použijeme GridSearchCV.

In [ ]:
# Definice pipeline pro optimalizaci
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kernel_approx', RBFSampler(random_state=42)),
    ('sgd', SGDClassifier(max_iter=1000, tol=1e-3, random_state=42))
])

# Definice parametrů pro GridSearch
param_grid = {
    'kernel_approx__gamma': [0.1, 0.5, 1.0, 2.0],
    'kernel_approx__n_components': [50, 100, 200],
    'sgd__alpha': [0.0001, 0.001, 0.01]
}

# Pro rychlejší zpracování použijeme subset dat (pokud je potřeba)
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)

print("Hledání optimálních hyperparametrů...")
grid_search.fit(X_moons_train, y_moons_train)
print("Hledání dokončeno.")

# Výsledky grid search
print(f"Nejlepší skóre: {grid_search.best_score_:.4f}")
print(f"Nejlepší parametry: {grid_search.best_params_}")

In [ ]:
# Vytvoření modelu s nejlepšími parametry
best_gamma = grid_search.best_params_['kernel_approx__gamma']
best_n_components = grid_search.best_params_['kernel_approx__n_components']
best_alpha = grid_search.best_params_['sgd__alpha']

best_model = make_pipeline(
    StandardScaler(),
    RBFSampler(gamma=best_gamma, n_components=best_n_components, random_state=42),
    SGDClassifier(alpha=best_alpha, max_iter=1000, tol=1e-3, random_state=42)
)

# Trénování a evaluace nejlepšího modelu
best_model.fit(X_moons_train, y_moons_train)
y_pred = best_model.predict(X_moons_test)
best_accuracy = accuracy_score(y_moons_test, y_pred)

print(f"Přesnost nejlepšího modelu: {best_accuracy:.4f}")
print("\nKlasifikační report:")
print(classification_report(y_moons_test, y_pred))

In [ ]:
# Vizualizace rozhodovací hranice nejlepšího modelu
plt.figure(figsize=(10, 6))
plot_decision_boundary(best_model, X_moons, y_moons, "Optimální Kernel Approximation model")
plt.show()

## 6. Škálovatelnost na větších datových sadách (simulace)

Jednou z hlavních výhod kernel approximation je škálovatelnost na velkých datových sadách. Simulujeme to generováním větších syntetických datasetů.

In [ ]:
# Simulace výkonu na datových sadách různých velikostí
sample_sizes = [1000, 5000, 10000, 20000, 50000]
scaling_results = {'rbf_sampler': [], 'svm_rbf': []}

for size in sample_sizes:
    print(f"Generování a trénování na datasetu velikosti {size}...")
    
    # Generování většího datasetu
    X, y = make_moons(n_samples=size, noise=0.1, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # RBFSampler s SGDClassifier
    rbf_sampler = make_pipeline(
        StandardScaler(),
        RBFSampler(gamma=2, n_components=100, random_state=42),
        SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
    )
    
    start_time = time()
    rbf_sampler.fit(X_train, y_train)
    rbf_sampler_time = time() - start_time
    rbf_sampler_accuracy = accuracy_score(y_test, rbf_sampler.predict(X_test))
    scaling_results['rbf_sampler'].append((size, rbf_sampler_time, rbf_sampler_accuracy))
    
    # SVM s RBF jádrem (pro menší velikosti)
    if size <= 10000:  # SVM je příliš pomalý pro velmi velké datasety
        svm_rbf = make_pipeline(
            StandardScaler(),
            SVC(kernel='rbf', gamma=2, random_state=42)
        )
        
        start_time = time()
        svm_rbf.fit(X_train, y_train)
        svm_time = time() - start_time
        svm_accuracy = accuracy_score(y_test, svm_rbf.predict(X_test))
        scaling_results['svm_rbf'].append((size, svm_time, svm_accuracy))
        
        print(f"  RBFSampler: čas = {rbf_sampler_time:.4f}s, přesnost = {rbf_sampler_accuracy:.4f}")
        print(f"  SVM RBF: čas = {svm_time:.4f}s, přesnost = {svm_accuracy:.4f}\n")
    else:
        print(f"  RBFSampler: čas = {rbf_sampler_time:.4f}s, přesnost = {rbf_sampler_accuracy:.4f}")
        print(f"  SVM RBF: Přeskočeno kvůli velikosti dat\n")
        scaling_results['svm_rbf'].append((size, None, None))  # Placeholder pro chybějící data

In [ ]:
# Vizualizace výsledků škálovatelnosti
rbf_sizes = [res[0] for res in scaling_results['rbf_sampler']]
rbf_times = [res[1] for res in scaling_results['rbf_sampler']]
rbf_accuracies = [res[2] for res in scaling_results['rbf_sampler']]

svm_sizes = [res[0] for res in scaling_results['svm_rbf'] if res[1] is not None]
svm_times = [res[1] for res in scaling_results['svm_rbf'] if res[1] is not None]
svm_accuracies = [res[2] for res in scaling_results['svm_rbf'] if res[2] is not None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Graf času trénování v závislosti na velikosti datasetu
ax1.plot(rbf_sizes, rbf_times, 'o-', label='RBFSampler + SGD', color='#2A9D8F')
ax1.plot(svm_sizes, svm_times, 'o-', label='SVM s RBF jádrem', color='#E76F51')
ax1.set_xlabel('Velikost datasetu')
ax1.set_ylabel('Čas trénování [s]')
ax1.set_title('Škálovatelnost: Čas trénování vs. Velikost dat')
ax1.legend()
ax1.grid(True)

# Graf přesnosti v závislosti na velikosti datasetu
ax2.plot(rbf_sizes, rbf_accuracies, 'o-', label='RBFSampler + SGD', color='#2A9D8F')
ax2.plot(svm_sizes, svm_accuracies, 'o-', label='SVM s RBF jádrem', color='#E76F51')
ax2.set_xlabel('Velikost datasetu')
ax2.set_ylabel('Přesnost')
ax2.set_title('Škálovatelnost: Přesnost vs. Velikost dat')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Demonstrace online učení

Další výhodou kernel approximation je možnost použít ho s modely podporujícími online učení.

In [ ]:
# Online učení s RBFSampler a SGDClassifier
# Generujeme větší dataset
X_large, y_large = make_moons(n_samples=20000, noise=0.1, random_state=42)
X_large_train, X_large_test, y_large_train, y_large_test = train_test_split(X_large, y_large, test_size=0.2, random_state=42)

# Transformace příznaků pomocí RBFSampler
scaler = StandardScaler()
X_large_train_scaled = scaler.fit_transform(X_large_train)
X_large_test_scaled = scaler.transform(X_large_test)

rbf_feature = RBFSampler(gamma=2, n_components=100, random_state=42)
X_large_train_rbf = rbf_feature.fit_transform(X_large_train_scaled)
X_large_test_rbf = rbf_feature.transform(X_large_test_scaled)

# Nastavení SGD pro online učení
sgd = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.001, 
                   learning_rate='optimal', warm_start=True, 
                   random_state=42)

# Rozdělení dat do dávek pro inkrementální učení
batch_size = len(X_large_train_rbf) // 10
n_batches = 10

online_accuracies = []

print("Začátek online učení...")

for i in range(n_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size
    
    # Trénování na aktuální dávce
    sgd.partial_fit(
        X_large_train_rbf[start_idx:end_idx], 
        y_large_train[start_idx:end_idx],
        classes=np.unique(y_large_train)
    )
    
    # Evaluace po každé dávce
    accuracy = sgd.score(X_large_test_rbf, y_large_test)
    online_accuracies.append(accuracy)
    
    print(f"Dávka {i+1}/{n_batches}: Přesnost = {accuracy:.4f}")

In [ ]:
# Vizualizace procesu online učení
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_batches + 1), online_accuracies, 'o-', color='#2A9D8F')
plt.xlabel('Počet zpracovaných dávek')
plt.ylabel('Přesnost na testovací sadě')
plt.title('Online učení s Kernel Approximation')
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Závěr

### Shrnutí poznatků o Kernel Approximation

V tomto notebooku jsme prozkoumali techniku kernel approximation, která umožňuje efektivní použití jádrových metod na velkých datových sadách.

**Klíčové poznatky:**

1. **Nelineární klasifikace** - Kernel approximation umožňuje lineárním modelům řešit nelineární problémy, což jsme demonstrovali na syntetických datových sadách (půlměsíce a kružnice).

2. **Škálovatelnost** - Zatímco tradiční jádrové metody jako SVM s RBF jádrem mají kvadratickou složitost, kernel approximation umožňuje lineární škálovatelnost, což je zásadní pro velké datové sady.

3. **Kompromis mezi přesností a rychlostí** - Počet komponent v RBFSampler představuje kompromis mezi přesností aproximace a výpočetní náročností. S rostoucím počtem komponent se přesnost zvyšuje, ale za cenu delšího času výpočtu.

4. **Metody aproximace** - Porovnali jsme dva přístupy ke kernel approximation: RBFSampler (založený na náhodných Fourierových příznacích) a Nystroem, přičemž každý má své silné stránky.

5. **Optimalizace hyperparametrů** - Pro dosažení nejlepších výsledků je důležité optimalizovat hyperparametry jako gamma (šířka jádra) a n_components (počet dimenzí aproximace).

6. **Online učení** - Kernel approximation umožňuje použití online učení, což je užitečné pro zpracování proudových dat nebo velmi velkých datasetů, které se nevejdou do paměti.

### Doporučení pro použití Kernel Approximation v praxi:

1. **Kdy použít:** Když máte velký dataset s nelineárními vzory a tradiční jádrové metody jsou výpočetně příliš náročné.

2. **Volba počtu komponent:** Začněte s menším počtem (např. 100) a postupně zvyšujte, dokud nedosáhnete požadované přesnosti nebo dokud se přesnost nepřestane zlepšovat.

3. **Optimalizace gamma:** Parametr gamma určuje šířku RBF jádra a má zásadní vliv na výsledek. Použijte cross-validaci pro nalezení optimální hodnoty.

4. **Předběžné zpracování:** Nezapomeňte na standardizaci dat před aplikací kernel approximation.

5. **Kombinace s efektivními lineárními modely:** Kernel approximation nejlépe funguje v kombinaci s rychlými lineárními modely jako SGD nebo Logistická regrese.

6. **Porovnání různých metod:** Experimentujte s různými metodami kernel approximation (RBFSampler vs. Nystroem) pro váš konkrétní problém.

Kernel approximation představuje efektivní způsob, jak využít sílu jádrových metod i na velkých datových sadách, a je cennou součástí nástrojů pro praktické strojové učení.